In [5]:
import glob
from pathlib import Path

import rasterio
from rasterio.io import MemoryFile
from rasterio.merge import merge
from rasterio.warp import Resampling, calculate_default_transform, reproject

# Web Mercator defined as a self-contained PROJ4 string.
# Using a full PROJ4 string instead of "EPSG:3857" means PROJ does not need
# to consult proj.db to resolve the CRS parameters.
_WEB_MERCATOR_PROJ4 = (
    "+proj=merc +a=6378137 +b=6378137 +lat_ts=0 +lon_0=0 "
    "+x_0=0 +y_0=0 +k=1 +units=m +nadgrids=@null +wktext +no_defs"
)


def merge_maps(outdir: Path) -> dict[str, Path]:
    """Merge all product maps in the output directory into .tif files.

    Returns a mapping of product name -> merged output path.
    """

    if not outdir.exists():
        raise FileNotFoundError(f"Output directory {outdir} does not exist.")

    # Find all .tif files in the output directory
    tifs = glob.glob(str(outdir / "*" / "*.tif"))
    if len(tifs) == 0:
        raise FileNotFoundError("No tif files found in the output directory to merge.")

    product_groups: dict[str, list[str]] = {}
    for tif in tifs:
        product = Path(tif).name.split("_")[0]
        product_groups.setdefault(product, []).append(tif)

    merged_outputs: dict[str, Path] = {}

    def _merge_tifs(product_name: str, product_tifs: list[str]) -> Path:
        reprojected_tifs = []
        dst_crs = rasterio.CRS.from_proj4(_WEB_MERCATOR_PROJ4)
        with rasterio.Env(CPL_LOG="ERROR"):
            for tif in product_tifs:
                with rasterio.open(tif) as src:
                    transform, width, height = calculate_default_transform(
                        src.crs, dst_crs, src.width, src.height, *src.bounds
                    )

                    kwargs = src.meta.copy()
                    kwargs.update(
                        {
                            "crs": dst_crs,
                            "transform": transform,
                            "width": width,
                            "height": height,
                        }
                    )

                    memfile = MemoryFile()
                    with memfile.open(**kwargs) as dst:
                        for i in range(1, src.count + 1):
                            reproject(
                                source=rasterio.band(src, i),
                                destination=rasterio.band(dst, i),
                                src_transform=src.transform,
                                src_crs=src.crs,
                                dst_transform=transform,
                                dst_crs=dst_crs,
                                resampling=Resampling.nearest,
                            )
                        dst.descriptions = src.descriptions
                    reprojected_tifs.append(memfile.open())

            # Merge all reprojected rasters
            mosaic, out_trans = merge(reprojected_tifs)

            # Use metadata from one of the input files and update
            out_meta = reprojected_tifs[0].meta.copy()
            out_meta.update(
                {
                    "driver": "GTiff",
                    "height": mosaic.shape[1],
                    "width": mosaic.shape[2],
                    "transform": out_trans,
                    "compress": "lzw",
                }
            )

            # Write to output
            outfile = outdir / f"{product_name}_merged.tif"
            with rasterio.open(outfile, "w", **out_meta) as dest:
                dest.write(mosaic)
                # Preserve band descriptions (if any)
                for idx, desc in enumerate(reprojected_tifs[0].descriptions, start=1):
                    if desc:
                        dest.set_band_description(idx, desc)

        return outfile

    for product_name, product_tifs in product_groups.items():
        merged_outputs[product_name] = _merge_tifs(product_name, product_tifs)

    return merged_outputs

In [6]:
from pathlib import Path

merge_maps(
    Path("/home/vverelst/Desktop/worldcereal_job_j-26052805261043e7a66a56ae77129e29/")
)

{'cropland-croptype': PosixPath('/home/vverelst/Desktop/worldcereal_job_j-26052805261043e7a66a56ae77129e29/cropland-croptype_merged.tif')}

In [4]:
import rasterio
import numpy as np

tif_path = "/home/vverelst/Desktop/worldcereal_soy_results/cropland-croptype_merged.tif"
band_index = 20  # S1 soy

with rasterio.open(tif_path) as src:
    band = src.read(
        band_index, masked=True
    )  # masks nodata automatically when available
    unique_values = np.unique(band.compressed())  # excludes masked/nodata pixels

print(unique_values)
print(f"Number of unique values: {len(unique_values)}")

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 254]
Number of unique values: 102
